# Module 1 Assignment — Question 4 — End-to-End Reproducibility Drill
### AI Operations (AIOps) — MLflow + DVC | Capstone

> Each partner runs their own local tracking server. The `sqlite` backend is required — the Model
> Registry does not work with a plain file store:
> ```bash
> mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000
> ```

## Step 0 — Setup

In [1]:
# !pip install mlflow scikit-learn pandas --quiet

import os
import json
import hashlib
import platform
import subprocess

import numpy as np
import mlflow
import mlflow.sklearn
import pandas as pd
import sklearn
from mlflow import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("q4-reproducibility-drill")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://localhost:5000


## Step 1 — Load the DVC-versioned dataset

In [2]:
SEED = 42
DATA_PATH = "data/mnist_784.npz"

with np.load(DATA_PATH) as npz:
    X = npz["X"] / 255.0
    y = npz["y"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"train={X_train.shape}  test={X_test.shape}")

train=(56000, 784)  test=(14000, 784)


## Step 2 — Collect provenance: git commit + data version

In [3]:
git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()

# "true" means there were uncommitted edits — the commit alone would not reproduce this run.
git_dirty = bool(subprocess.run(
    ["git", "status", "--porcelain"], capture_output=True, text=True
).stdout.strip())

# Hash the bytes we actually trained on, and pick up the md5 DVC recorded for the same file.
data_md5 = hashlib.md5(open(DATA_PATH, "rb").read()).hexdigest()

# `dvc add data` tracks the whole directory, so the pointer is data.dvc, not <file>.dvc.
dvc_file = DATA_PATH + ".dvc"
if not os.path.exists(dvc_file):
    dvc_file = "data.dvc"

dvc_md5 = None
try:
    for line in open(dvc_file):
        # a dir-level pointer writes "- md5: ...", a file-level one writes "md5: ..."
        entry = line.strip().lstrip("- ")
        if entry.startswith("md5:"):
            dvc_md5 = entry.split("md5:")[1].strip()
            break
except FileNotFoundError:
    print(f"WARNING: {dvc_file} not found — run `dvc add {DATA_PATH}` before committing.")

print(f"git_commit={git_commit}  dirty={git_dirty}")
print(f"data_md5={data_md5}  dvc_md5={dvc_md5}")

git_commit=06c519089bbd3586130f50784cf12110e2d3efed  dirty=True
data_md5=342ecb5219278ec13194d5bb2b77518a  dvc_md5=d3b391f5e30a5b6843e51f4d76e6d75b.dir


## Step 3 — One instrumented run

In [4]:
MODEL_NAME = "mnist-mlp-q4"
BASELINE_PATH = "sujal_run.json"
TOLERANCE = 0.005

# The first run writes the baseline file; a clone that already has it is a reproduction.
is_baseline = not os.path.exists(BASELINE_PATH)

with mlflow.start_run(run_name="baseline" if is_baseline else "reproduction") as run:
    hidden_layer_sizes = (128, 64)
    learning_rate_init = 0.001
    batch_size = 128
    max_iter = 20

    mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
    mlflow.log_param("learning_rate_init", learning_rate_init)
    mlflow.log_param("batch_size", batch_size)
    mlflow.log_param("max_iter", max_iter)
    mlflow.log_param("early_stopping", True)
    mlflow.log_param("seed", SEED)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("dataset", DATA_PATH)

    mlflow.set_tag("git_commit", git_commit)
    mlflow.set_tag("git_dirty", git_dirty)
    mlflow.set_tag("dataset_md5", data_md5)
    mlflow.set_tag("dataset_dvc_md5", dvc_md5)
    mlflow.set_tag("role", "baseline" if is_baseline else "reproduction")
    mlflow.set_tag("team", "data-science")
    mlflow.set_tag("sklearn_version", sklearn.__version__)
    mlflow.set_tag("python_version", platform.python_version())

    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate_init,
        batch_size=batch_size,
        max_iter=max_iter,
        early_stopping=True,
        random_state=SEED,
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    train_acc = accuracy_score(y_train, model.predict(X_train))

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_macro", f1)
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("generalization_gap", train_acc - acc)

    for epoch, (train_loss, val_acc) in enumerate(zip(model.loss_curve_, model.validation_scores_)):
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_acc, step=epoch)

    # Artifacts: the report Partner B compares against, and the .dvc pointer to the exact data.
    mlflow.log_text(classification_report(y_test, preds, digits=4), "reports/classification_report.txt")
    mlflow.log_text(
        pd.DataFrame(confusion_matrix(y_test, preds)).to_csv(index=False),
        "reports/confusion_matrix.csv",
    )
    mlflow.log_dict(
        {"accuracy": acc, "f1_macro": f1, "seed": SEED,
         "git_commit": git_commit, "dataset_md5": data_md5, "dataset_dvc_md5": dvc_md5},
        "reports/metrics.json",
    )
    try:
        mlflow.log_artifact(dvc_file, artifact_path="data_version")
    except FileNotFoundError:
        pass

    mlflow.sklearn.log_model(
        model,
        name="model",
        serialization_format="cloudpickle",
        input_example=X_test[:5],
        registered_model_name=MODEL_NAME,
    )

    run_id = run.info.run_id
    print(f"Logged run {run_id}  |  acc={acc:.4f}  f1={f1:.4f}  train_acc={train_acc:.4f}")

/home/sujal/miniconda3/envs/aiops/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/08/30 21:53:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'mnist-mlp-q4' already exists. Creating a new version of this model...
2026/08/30 21:53:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mnist-mlp-q4, version 7
Created version '7' of model 'mnist-mlp-q4'.


Logged run b0461ee06eca45c19b143b59268d0b33  |  acc=0.9751  f1=0.9750  train_acc=0.9962
🏃 View run baseline at: http://localhost:5000/#/experiments/7/runs/b0461ee06eca45c19b143b59268d0b33
🧪 View experiment at: http://localhost:5000/#/experiments/7


## Step 4 — Register the model and transition it to `Staging`

In [5]:
client = MlflowClient()

version = client.search_model_versions(f"run_id='{run_id}'")[0].version
client.update_model_version(MODEL_NAME, version, description=f"Q4 {run.info.run_name} — accuracy={acc:.4f}")
client.set_model_version_tag(MODEL_NAME, version, "validation_accuracy", f"{acc:.4f}")

try:
    client.transition_model_version_stage(MODEL_NAME, version, stage="Staging")
    print(f"{MODEL_NAME} v{version} → stage 'Staging'")
except Exception as exc:
    # MLflow 3 servers may have replaced stages with aliases — same intent, newer API.
    print(f"Stages unavailable ({exc.__class__.__name__}); using an alias instead.")
    client.set_registered_model_alias(MODEL_NAME, "staging", version)
    client.set_model_version_tag(MODEL_NAME, version, "stage", "Staging")
    print(f"{MODEL_NAME} v{version} → alias 'staging'")

mnist-mlp-q4 v7 → stage 'Staging'


/tmp/ipykernel_170786/2822199393.py:8: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(MODEL_NAME, version, stage="Staging")


## Step 5 — Confirm what was logged

In [6]:
runs_df = mlflow.search_runs(
    experiment_names=["q4-reproducibility-drill"],
    order_by=["attributes.start_time DESC"],
)

display_cols = [c for c in runs_df.columns if c in (
    "run_id", "tags.mlflow.runName", "params.seed", "params.learning_rate_init", "params.batch_size",
    "metrics.accuracy", "metrics.f1_macro", "tags.git_commit", "tags.dataset_md5",
)]
print(runs_df[display_cols].head(5).to_string(index=False))

print(f"\nrun_id: {run_id}")
print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}")

                          run_id  metrics.accuracy  metrics.f1_macro params.batch_size params.learning_rate_init params.seed tags.mlflow.runName                 tags.dataset_md5                          tags.git_commit
b0461ee06eca45c19b143b59268d0b33          0.975143          0.974968               128                     0.001          42            baseline 342ecb5219278ec13194d5bb2b77518a 06c519089bbd3586130f50784cf12110e2d3efed
766daa0e0e784f9c8b88a9d26af2a73a          0.975143          0.974968               128                     0.001          42            baseline 342ecb5219278ec13194d5bb2b77518a 06c519089bbd3586130f50784cf12110e2d3efed
c04d680ce9af4ca783dc2fe9d6105c0b          0.969571          0.969393               128                     0.001          42            baseline 342ecb5219278ec13194d5bb2b77518a 06c519089bbd3586130f50784cf12110e2d3efed

run_id: b0461ee06eca45c19b143b59268d0b33
accuracy=0.9751  f1_macro=0.9750


## Step 6 — Baseline record and reproduction check

In [7]:
record = {
    "run_id": run_id,
    "seed": SEED,
    "git_commit": git_commit,
    "dataset_md5": data_md5,
    "dataset_dvc_md5": dvc_md5,
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "tolerance": TOLERANCE,
    "params": {
        "hidden_layer_sizes": list(hidden_layer_sizes),
        "learning_rate_init": learning_rate_init,
        "batch_size": batch_size,
        "max_iter": max_iter,
        "test_size": 0.2,
    },
    "metrics": {"accuracy": acc, "f1_macro": f1, "train_accuracy": train_acc},
}

# Re-check the file here too, so a stale is_baseline from an earlier run cannot crash this cell.
if is_baseline or not os.path.exists(BASELINE_PATH):
    with open(BASELINE_PATH, "w") as fh:
        json.dump(record, fh, indent=2)
    print(f"Wrote {BASELINE_PATH} — commit it in the same commit as the .dvc file.")
else:
    with open(BASELINE_PATH) as fh:
        baseline = json.load(fh)

    tol = baseline.get("tolerance", TOLERANCE)
    lines = [
        "## Reproduction check",
        "",
        f"- Baseline run: `{baseline['run_id']}`",
        f"- This run: `{run_id}`",
        f"- Stated tolerance: +/- {tol} absolute",
        "",
        "| metric | baseline | this run | delta | within tolerance |",
        "| --- | --- | --- | --- | --- |",
    ]

    matched = True
    for name in ("accuracy", "f1_macro"):
        base_val = baseline["metrics"][name]
        this_val = record["metrics"][name]
        delta = this_val - base_val
        ok = abs(delta) <= tol
        matched &= ok
        lines.append(f"| {name} | {base_val:.4f} | {this_val:.4f} | {delta:+.4f} | {'yes' if ok else 'NO'} |")
        client.set_tag(run_id, f"repro_delta_{name}", f"{delta:+.6f}")

    # Anything different here explains a mismatch, so record it either way.
    lines += ["", "### Environment and data", ""]
    for key in ("dataset_md5", "git_commit", "sklearn_version", "python_version", "seed"):
        base_v, this_v = baseline.get(key), record.get(key)
        lines.append(f"- {key}: baseline=`{base_v}` this=`{this_v}` — {'match' if base_v == this_v else '**DIFFERENT**'}")

    verdict = "REPRODUCED" if matched else "NOT REPRODUCED"
    lines.insert(1, f"\n**Verdict: {verdict}** (tolerance +/- {tol}).\n")
    if not matched:
        lines += [
            "",
            "A different `dataset_md5` means `dvc checkout` did not restore the baseline data version; "
            "a different `sklearn_version` means the environment was not built from `environment.yml`.",
        ]

    note = "\n".join(lines)
    client.set_tag(run_id, "mlflow.note.content", note)
    client.set_tag(run_id, "repro_verdict", verdict)
    client.set_tag(run_id, "repro_tolerance", str(tol))
    client.set_tag(run_id, "repro_baseline_run", baseline["run_id"])
    print(note)

Wrote sujal_run.json — commit it in the same commit as the .dvc file.
